#Setup

In [1]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
EXCHANGE_API_KEY = userdata.get('EXCHANGE_API_KEY')

#Install the library

In [3]:
!pip install -q langchain-groq==1.0.0 langchain-core==1.0.4 requests==2.32.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
langchain 1.2.15 requires langchain-core<2.0.0,>=1.2.10, but you have langchain-core 1.0.4 which is incompatible.


#check the version

In [4]:
!pip show langchain-groq langchain-core requests

Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-core
Version: 1.0.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions
Required-by: langchain, langchain-groq, langgraph, langgraph-checkpoint, langgraph-prebuilt
---
Name: requests
Version: 2.32.5
Summary: Python HTTP for Humans.
Home-page: https://requests.readthedocs.io
Author: Kenneth Reitz
Author-email: me@kennethreitz.org
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: certifi, charset_normalizer, idna, u

#Import libraries from tool building

In [6]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

In [13]:
@tool
def multiply(a:int, b:int) -> int:
  """Given 2 numbers a and b this function returns their product"""
  return a * b


In [14]:
multiply.invoke({'a': 15,'b': 5})

75

In [15]:
multiply.name

'multiply'

In [16]:
multiply.description

'Given 2 numbers a and b this function returns their product'

In [17]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [22]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = GROQ_API_KEY,
    temperature = 0,
)

In [23]:
llm.invoke("Hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says "Hello". We need to respond appropriately. No special instructions. Just greet back.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 72, 'total_tokens': 111, 'completion_time': 0.081656787, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.002744697, 'prompt_tokens_details': None, 'queue_time': 0.004762824, 'total_time': 0.084401484}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_068241849b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--8864c92a-285c-4466-acbd-a49afdf21a61-0', usage_metadata={'input_tokens': 72, 'output_tokens': 39, 'total_tokens': 111})

In [29]:
llm_with_tools = llm.bind_tools([multiply])

In [25]:
llm_with_tools.invoke("How are you?")

AIMessage(content="I'm doing great, thank you! How can I assist you today?", additional_kwargs={'reasoning_content': 'The user asks "How are you?" It\'s a casual greeting. Should respond politely. No need for function calls.'}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 133, 'total_tokens': 180, 'completion_time': 0.096760843, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.005082577, 'prompt_tokens_details': None, 'queue_time': 0.076710106, 'total_time': 0.10184342}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_74e589d00d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--86e698c9-0655-4beb-ba3e-4687c4fe9838-0', usage_metadata={'input_tokens': 133, 'output_tokens': 47, 'total_tokens': 180})

In [35]:
query = llm_with_tools.invoke("What is 15 multiplied by 15?")
response = print(query)
response

content='' additional_kwargs={'reasoning_content': 'User asks: "What is 15 multiplied by 15?" Simple multiplication: 225. Could answer directly. No need to call function, but could demonstrate using function. Could call multiply with a=15,b=15. Let\'s do that.', 'tool_calls': [{'id': 'fc_f26b3082-518f-4319-a574-43fb029e7c4a', 'function': {'arguments': '{"a":15,"b":15}', 'name': 'multiply'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 138, 'total_tokens': 222, 'completion_time': 0.179020583, 'completion_tokens_details': {'reasoning_tokens': 51}, 'prompt_time': 0.005169451, 'prompt_tokens_details': None, 'queue_time': 0.005095061, 'total_time': 0.184190034}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--081f4b64-92dd-446b-82f2-9e3fa1cd30f2-0' tool_calls=[{'name': 'multiply', 'args': {'a': 1

In [36]:
llm.invoke("What is 1 USD in INR?")

AIMessage(content='I’m not able to provide real‑time exchange‑rate information. For the most accurate and up‑to‑date conversion of\u202f1\u202fUSD to INR, please check a reliable financial source such as:\n\n- A reputable currency‑conversion website (e.g., XE, OANDA, Bloomberg)\n- Your bank’s online portal or mobile app\n- A financial news service or market data platform\n\nThese sources update rates continuously and will give you the current value.', additional_kwargs={'reasoning_content': 'The user asks: "What is 1 USD in INR?" This is a request for current exchange rate. According to policy, providing real-time financial data is disallowed because we cannot guarantee up-to-date info. We must respond with a disclaimer that we cannot provide real-time data and suggest checking reliable sources. Also we can give a general approach. So we must refuse to provide the exact rate, but can give guidance.'}, response_metadata={'token_usage': {'completion_tokens': 188, 'prompt_tokens': 79, 'to

In [38]:
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This tool fetches the currency conversion factor between
   a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/{EXCHANGE_API_KEY}/pair/{base_currency}/{target_currency}'

  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate) -> float:
  """
  Given a conversion rate this function calculates the
  target currency value from a given base currency value
  """
  return base_currency_value * conversion_rate




In [39]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    api_key = GROQ_API_KEY,
    temperature = 0,
)

In [40]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [41]:
llm_with_tools.invoke("What is the conversion factor between USD and INR")

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide conversion factor. Use function get_conversion_factor.', 'tool_calls': [{'id': 'fc_35d5054e-1692-4912-aad6-326daaba5e9c', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 191, 'total_tokens': 243, 'completion_time': 0.1091541, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.008708633, 'prompt_tokens_details': None, 'queue_time': 0.004806004, 'total_time': 0.117862733}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6b677c2caf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--aa9a989f-2df2-4292-b8bd-5dbb4c6a2475-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'fc_35d5054e-1